# Generic MIMIC-like ingestion: NWICU

This lightweight example uses ConMedRL's reviewed declarative NWICU profile. The profile contains paths, verified local item IDs, unit conversions, and airway event rules—no site-specific Python adapter and no patient data.

In [ ]:
import os
from pathlib import Path

from ConMedRL.data import ImputationConfig, PreprocessConfig, build_dataset
from ConMedRL.data.profiles import load_nwicu_profile

DATA_DIR = Path(os.environ.get("NWICU_DATA_DIR", "../icu_data/nwicu"))
TASK = "extubation"  # or "discharge"
RUN_PIPELINE = False  # the full local extraction can take several minutes

dataset_spec, task_spec, approval_hash = load_nwicu_profile(TASK)
print("Profile:", dataset_spec["name"], "task:", task_spec["name"])
print("Profile warnings:", dataset_spec.get("warnings", []) + task_spec.get("warnings", []))

if RUN_PIPELINE:
    # Fingerprinting the selected local files makes approval content-addressed.
    dataset_spec, task_spec, approval_hash = load_nwicu_profile(
        TASK, data_dir=DATA_DIR
    )
    config = PreprocessConfig(
        database="generic",
        task=TASK,
        data_dir=DATA_DIR,
        output_dir="./processed_nwicu",
        output_formats=("csv", "parquet", "d3rlpy"),
        dataset_spec=dataset_spec,
        task_spec=task_spec,
        approved_plan_hash=approval_hash,
        min_variable_coverage=0.001,
        imputation=ImputationConfig(knn_impute=False),
    )
    bundle = build_dataset(config)
    print(bundle.summary())
    print("Resolved states:", bundle.schema.names)
    print("Action:", bundle.action_type, bundle.action_columns)
    print("Constraints:", bundle.num_constraints)

## Validation note

The reviewed local NWICU profile was exercised without a full RL experiment. Discharge produced 30,095 episodes and a dynamic 26-variable state space. Point-event intubation/ventilation→extubation pairing produced 1,961 extubation episodes and a dynamic 31-variable state space. Both generated patient-disjoint train/validation/test splits. Exact dimensions can change with cohort settings and source coverage; always use `bundle.state_dim`.